# Full MTS + Mistral 7B Instruct on ASAP, paper-style split

Bản này chuyển từ **MTS-lite 1-turn** sang **Full MTS 2-turn**:

- Turn 1: quote retrieval theo từng trait
- Turn 2: scoring theo từng trait dựa trên quote đã lấy
- Aggregate: average trait scores
- Scaling: outlier clipping + min-max scaling theo từng prompt, giống mô tả trong paper

Lưu ý: MTS là **zero-shot**, nên không có bước train/fine-tune model. “Chia y hệt paper” ở đây nghĩa là:
- gom toàn bộ ASAP lại
- random sample **10% essays của mỗi prompt** làm test set
- chỉ chạy LLM trên test set đó để tính QWK

Paper không công bố random seed / danh sách essay_id test, nên notebook này tái hiện **đúng protocol**, nhưng không thể đảm bảo trùng đúng từng essay với paper.


In [1]:
# =========================
# 0. Check GPU
# =========================
!nvidia-smi


Sat May 16 16:18:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P0             69W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
# =========================
# 1. Install dependencies
# =========================
%pip install -q -U "transformers==4.51.3" "accelerate==1.6.0" "bitsandbytes==0.45.5" "sentencepiece==0.2.0" "tqdm==4.67.1" "openpyxl==3.1.5"


In [3]:
# =========================
# 2. Imports
# =========================
import os, re, gc, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("torch:", torch.__version__)
a = np.array(["abc", "def"])
print("numpy char test:", np.char.center(a, 8).tolist())


numpy: 2.0.2
pandas: 2.2.2
torch: 2.10.0+cu128
numpy char test: ['  abc   ', '  def   ']


In [4]:
# =========================
# 3. Config
# =========================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Option A: nếu có full ASAP file, đặt path ở đây.
# Ví dụ Kaggle ASAP thường là training_set_rel3.tsv.
ASAP_FULL_PATH = "/content/training_set_rel3.tsv"

# Option B: nếu bạn đang có 3 file split cũ, notebook sẽ concat lại rồi sample 10% mỗi prompt.
TRAIN_PATH = "/content/asap_train.csv"
VAL_PATH   = "/content/asap_val.csv"
TEST_PATH  = "/content/asap_test.csv"

OUTPUT_DIR = Path("/content/mts_mistral7b_full_paper_split_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

# Paper-style ASAP split
PAPER_TEST_FRACTION = 0.10
SPLITS_TO_SCORE = ["test"]  # MTS zero-shot: chỉ score test set
PROMPTS_TO_SCORE = None     # ví dụ [1,2] để debug
MAX_ESSAYS_PER_PROMPT_PER_SPLIT = 1000  # ví dụ 20 để smoke test trước

# Inference settings
INFER_BATCH_SIZE = 48
USE_4BIT = True
TEMPERATURE = 0.1
REPETITION_PENALTY = 1.1
MAX_NEW_TOKENS_QUOTE = 384
MAX_NEW_TOKENS_SCORE = 96
MAX_INPUT_LENGTH = 4096

print("Output dir:", OUTPUT_DIR)
print("Model:", MODEL_NAME)
print("Paper test fraction:", PAPER_TEST_FRACTION)
print("Batch size:", INFER_BATCH_SIZE)


Output dir: /content/mts_mistral7b_full_paper_split_outputs
Model: mistralai/Mistral-7B-Instruct-v0.2
Paper test fraction: 0.1
Batch size: 48


In [5]:
# =========================
# 4. Score ranges, prompt placeholders, and prompt-specific traits
# =========================
ASAP_SCORE_RANGES = {
    1: (2, 12), 2: (1, 6), 3: (0, 3), 4: (0, 3),
    5: (0, 4), 6: (0, 4), 7: (0, 30), 8: (0, 60),
}

ASAP_PROMPTS = {
    1: "Write a persuasive essay explaining whether computers are beneficial or harmful to people.",
    2: "Write a persuasive essay explaining whether censorship in libraries is acceptable.",
    3: "Write a response based on the provided source text.",
    4: "Write a response based on the provided source text.",
    5: "Write a response based on the provided source text.",
    6: "Write a response based on the provided source text.",
    7: "Write a narrative essay based on the given story prompt.",
    8: "Write a narrative essay based on the given story prompt.",
}

PROMPT_TRAITS = {
    1: [
        {
            "trait": "Position and Thesis Clarity",
            "description": "This dimension evaluates how clearly the writer takes a stance on the effects of computers on people and how effectively this stance is conveyed.",
            "rubric": """0-2: The position is unclear or absent; the thesis lacks a clear stance or is missing.
3-5: The position is somewhat evident but lacks clarity or specificity.
6-8: The position is clear, though it may need more specificity or nuance.
9-10: The position is very clear and the thesis effectively communicates the writer's stance with precision."""
        },
        {
            "trait": "Supporting Details and Evidence",
            "description": "This dimension assesses the quality, relevance, and specificity of supporting details used to back the writer's position.",
            "rubric": """0-2: Very minimal or no supporting details are provided.
3-5: Supporting details are general, vague, repetitive, or weakly related to the thesis.
6-8: Adequate supporting details are provided, though some may lack specificity or depth.
9-10: Rich, specific, and relevant details strongly support the thesis."""
        },
        {
            "trait": "Organization and Structure",
            "description": "This dimension evaluates coherence, logical progression, paragraphing, and overall essay structure.",
            "rubric": """0-2: The essay lacks organization and is difficult to follow.
3-5: The essay shows minimal organization but lacks coherent structure or clear transitions.
6-8: The essay is generally organized with some coherence and progression.
9-10: The essay is strongly organized with clear transitions and logical development."""
        },
        {
            "trait": "Style, Language, and Audience Awareness",
            "description": "This dimension assesses language use, style, clarity, and the writer's ability to engage the intended audience.",
            "rubric": """0-2: Language is awkward or unclear, with little awareness of audience.
3-5: Language is basic and engagement with the audience is limited.
6-8: Language is generally clear and sometimes engaging, with some awareness of audience.
9-10: Language is engaging, precise, and consistently appropriate for the audience."""
        },
    ],
    2: [
        {
            "trait": "Position and Argument Focus",
            "description": "This dimension evaluates how clearly the essay presents a position on the issue and maintains focus on that argument.",
            "rubric": """0-2: The position is absent, unclear, or unrelated to the prompt.
3-5: The position is present but vague, inconsistent, or weakly maintained.
6-8: The position is clear and mostly maintained throughout the essay.
9-10: The position is clear, focused, and consistently developed with strong control."""
        },
        {
            "trait": "Reasons and Persuasive Support",
            "description": "This dimension evaluates the relevance and persuasiveness of reasons, examples, and explanations.",
            "rubric": """0-2: Little or no relevant support is provided.
3-5: Support is limited, generic, repetitive, or only partly persuasive.
6-8: Reasons and examples are mostly relevant and adequately developed.
9-10: Reasons and examples are specific, convincing, and effectively explained."""
        },
        {
            "trait": "Organization and Coherence",
            "description": "This dimension evaluates essay structure, sequencing of ideas, paragraphing, and transitions.",
            "rubric": """0-2: Ideas are disorganized and difficult to follow.
3-5: Some organization is present, but progression is uneven or transitions are weak.
6-8: The essay is generally coherent with logical progression and some transitions.
9-10: The essay is well organized, coherent, and smoothly connected throughout."""
        },
        {
            "trait": "Language Control and Conventions",
            "description": "This dimension evaluates sentence control, vocabulary, grammar, spelling, punctuation, and clarity.",
            "rubric": """0-2: Errors severely interfere with meaning.
3-5: Frequent errors or limited vocabulary sometimes interfere with clarity.
6-8: Language is generally clear with some variety and mostly controlled errors.
9-10: Language is fluent, precise, and controlled with few errors."""
        },
    ],
}

SOURCE_DEPENDENT_TRAITS = [
    {
        "trait": "Source Comprehension",
        "description": "This dimension evaluates whether the response accurately understands the source text and the prompt requirements.",
        "rubric": """0-2: The response shows little or no accurate understanding of the source or task.
3-5: The response shows partial understanding with important omissions or inaccuracies.
6-8: The response shows mostly accurate understanding of key ideas.
9-10: The response shows accurate, thorough, and task-appropriate understanding."""
    },
    {
        "trait": "Use of Relevant Textual Evidence",
        "description": "This dimension evaluates how effectively the response uses relevant details from the source.",
        "rubric": """0-2: Source support is absent, irrelevant, or mostly incorrect.
3-5: Evidence is limited, vague, copied, or weakly connected to the task.
6-8: Evidence is relevant and mostly well connected to the answer.
9-10: Evidence is specific, accurate, well selected, and effectively explained."""
    },
    {
        "trait": "Response Focus and Development",
        "description": "This dimension evaluates whether the response fully answers the question with clear development.",
        "rubric": """0-2: The response is minimal, off-task, or undeveloped.
3-5: The response partially addresses the task but is underdeveloped or unfocused.
6-8: The response addresses the task with adequate focus and development.
9-10: The response fully addresses the task with clear, complete, and effective development."""
    },
    {
        "trait": "Organization and Language Control",
        "description": "This dimension evaluates clarity, organization, sentence control, grammar, spelling, and punctuation.",
        "rubric": """0-2: The response is very hard to follow due to organization or language problems.
3-5: Some organization is present but language errors reduce clarity.
6-8: The response is generally organized and understandable with some errors.
9-10: The response is well organized, clear, and well controlled."""
    },
]

PROMPT_TRAITS[3] = SOURCE_DEPENDENT_TRAITS
PROMPT_TRAITS[4] = SOURCE_DEPENDENT_TRAITS
PROMPT_TRAITS[5] = SOURCE_DEPENDENT_TRAITS
PROMPT_TRAITS[6] = SOURCE_DEPENDENT_TRAITS

PROMPT_TRAITS[7] = [
    {"trait": "Narrative Focus and Story Development", "description": "This dimension evaluates whether the narrative is focused and develops a clear story.", "rubric": """0-2: The narrative is minimal, unclear, or lacks a story.
3-5: A story is present but underdeveloped, confusing, or weakly focused.
6-8: The story is mostly clear and adequately developed.
9-10: The story is clear, engaging, and well developed."""},
    {"trait": "Plot Structure and Organization", "description": "This dimension evaluates beginning, middle, end, sequencing, pacing, and transitions.", "rubric": """0-2: Events are disorganized or very difficult to follow.
3-5: Events have some sequence but organization or pacing is weak.
6-8: Events are generally organized with a clear sequence.
9-10: Events are well structured with effective pacing and transitions."""},
    {"trait": "Descriptive Detail and Elaboration", "description": "This dimension evaluates the use of details, elaboration, characters, setting, and sensory description.", "rubric": """0-2: Little or no detail or elaboration is provided.
3-5: Details are limited, general, or repetitive.
6-8: Details adequately develop events, characters, or setting.
9-10: Rich and specific details make the narrative vivid and engaging."""},
    {"trait": "Language Use and Conventions", "description": "This dimension evaluates word choice, sentence variety, grammar, spelling, punctuation, and readability.", "rubric": """0-2: Language problems severely interfere with meaning.
3-5: Frequent errors or simple language limit clarity and effect.
6-8: Language is generally clear with some variety and manageable errors.
9-10: Language is fluent, vivid, and well controlled."""},
]

PROMPT_TRAITS[8] = [
    {"trait": "Narrative Purpose and Development", "description": "This dimension evaluates how fully the response develops a meaningful narrative based on the prompt.", "rubric": """0-2: The response is minimal, off-task, or lacks narrative development.
3-5: The narrative is present but thin, inconsistent, or underdeveloped.
6-8: The narrative is clear and adequately developed.
9-10: The narrative is rich, purposeful, and thoroughly developed."""},
    {"trait": "Organization, Sequence, and Coherence", "description": "This dimension evaluates logical sequencing, paragraphing, pacing, and coherence across the story.", "rubric": """0-2: The story is disorganized and difficult to follow.
3-5: The story has some sequence but weak coherence or pacing.
6-8: The story is mostly coherent with clear sequence and adequate pacing.
9-10: The story is well organized, coherent, and smoothly paced."""},
    {"trait": "Elaboration, Voice, and Descriptive Detail", "description": "This dimension evaluates elaboration of events, voice, imagery, characters, setting, and narrative detail.", "rubric": """0-2: Very little elaboration or descriptive detail is present.
3-5: Details are limited, generic, or uneven.
6-8: Details and voice adequately support the narrative.
9-10: Strong voice and vivid details make the narrative engaging and memorable."""},
    {"trait": "Sentence Fluency and Conventions", "description": "This dimension evaluates sentence control, variety, grammar, spelling, punctuation, and overall readability.", "rubric": """0-2: Severe errors make the response difficult to understand.
3-5: Frequent errors or awkward sentences reduce readability.
6-8: Sentences are generally clear with some variety and manageable errors.
9-10: Sentences are fluent and varied, with strong control of conventions."""},
]

def get_traits(row):
    return PROMPT_TRAITS[int(row["essay_set"])]

In [6]:
# =========================
# 5. Load ASAP and create paper-style split
# =========================
def read_asap_file(path):
    path = Path(path)
    if not path.exists():
        return None

    if path.suffix.lower() == ".tsv":
        df = pd.read_csv(path, sep="\t", encoding="latin1")
    else:
        df = pd.read_csv(path)

    # Normalize common column names.
    rename = {}
    if "essay_set" not in df.columns and "prompt" in df.columns:
        rename["prompt"] = "essay_set"
    if "domain1_score" not in df.columns and "score" in df.columns:
        rename["score"] = "domain1_score"
    if rename:
        df = df.rename(columns=rename)

    required = ["essay_id", "essay_set", "essay", "domain1_score"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{path} missing columns {missing}. Found: {df.columns.tolist()}")

    keep_cols = required + [c for c in ["prompt_text"] if c in df.columns]
    df = df[keep_cols].copy()
    df["essay_id"] = df["essay_id"].astype(int)
    df["essay_set"] = df["essay_set"].astype(int)
    df["essay"] = df["essay"].astype(str)
    df["domain1_score"] = df["domain1_score"].round().astype(int)
    return df

def load_full_asap():
    # Prefer official full ASAP if available.
    full = read_asap_file(ASAP_FULL_PATH)
    if full is not None:
        print("Loaded full ASAP file:", ASAP_FULL_PATH, full.shape)
        return full

    # Otherwise reconstruct full data by concatenating existing train/val/test files.
    pieces = []
    for p in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
        df = read_asap_file(p)
        if df is not None:
            print("Loaded:", p, df.shape)
            pieces.append(df)

    if not pieces:
        raise FileNotFoundError(
            "Cannot find ASAP_FULL_PATH or any of TRAIN_PATH/VAL_PATH/TEST_PATH. "
            "Upload training_set_rel3.tsv or asap_train.csv/asap_val.csv/asap_test.csv to /content."
        )

    full = pd.concat(pieces, ignore_index=True)
    full = full.drop_duplicates(subset=["essay_id"], keep="first").reset_index(drop=True)
    print("Reconstructed full ASAP from split files:", full.shape)
    return full

full_df = load_full_asap()

# Paper protocol: randomly sample 10% essays from each ASAP prompt for test.
test_parts = []
trainval_parts = []
for essay_set, g in full_df.groupby("essay_set", sort=True):
    n_test = int(round(len(g) * PAPER_TEST_FRACTION))
    n_test = max(1, n_test)
    test_g = g.sample(n=n_test, random_state=SEED)
    trainval_g = g.drop(test_g.index)
    test_parts.append(test_g)
    trainval_parts.append(trainval_g)

paper_test_df = pd.concat(test_parts, ignore_index=True)
paper_trainval_df = pd.concat(trainval_parts, ignore_index=True)

paper_test_df["split"] = "test"
paper_trainval_df["split"] = "trainval_unused"

# MTS is zero-shot, so trainval is not used for fitting. Kept only for checking.
all_df = pd.concat([paper_trainval_df, paper_test_df], ignore_index=True)

print("Full ASAP:", full_df.shape)
print("Paper-style test:", paper_test_df.shape)
print("Unused trainval:", paper_trainval_df.shape)

display(
    all_df.groupby(["split", "essay_set"])["domain1_score"]
    .agg(["count", "min", "max", "mean"])
    .reset_index()
)

# =========================
# 6. Select rows to score
# =========================
score_df = all_df[all_df["split"].isin(SPLITS_TO_SCORE)].copy()

if PROMPTS_TO_SCORE is not None:
    score_df = score_df[score_df["essay_set"].isin(PROMPTS_TO_SCORE)].copy()

if MAX_ESSAYS_PER_PROMPT_PER_SPLIT is not None:
    pieces = []
    for (split, essay_set), g in score_df.groupby(["split", "essay_set"]):
        n = min(MAX_ESSAYS_PER_PROMPT_PER_SPLIT, len(g))
        pieces.append(g.sample(n=n, random_state=SEED))
    score_df = pd.concat(pieces, ignore_index=True)

score_df = score_df.sort_values(["split", "essay_set", "essay_id"]).reset_index(drop=True)

print("Rows to score:", len(score_df))
display(score_df.groupby(["split", "essay_set"])["essay_id"].count().reset_index(name="n"))

# =========================
# 7. Load Mistral 7B
# =========================
if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )
else:
    bnb_config = None

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    quantization_config=bnb_config,
)
model.eval()

print("Model loaded:", MODEL_NAME)
print("pad_token:", tokenizer.pad_token)

# =========================
# 8. Full MTS prompt builders: quote retrieval + scoring
# =========================
ASAP_NOTE = """I have made an effort to remove personally identifying information from the essays using the Named Entity Recognizer (NER). The relevant entities are identified in the text and then replaced with strings such as "@PERSON", "@ORGANIZATION", "@LOCATION", "@DATE", "@TIME", "@MONEY", "@PERCENT", "@CAPS", and "@NUM". Please do not penalize the essay because of the anonymizations."""

def get_prompt_text(row):
    if "prompt_text" in row.index and isinstance(row["prompt_text"], str) and row["prompt_text"].strip():
        return row["prompt_text"]
    return ASAP_PROMPTS[int(row["essay_set"])]

def build_quote_messages(prompt_text, essay, trait_obj):
    trait = trait_obj["trait"]
    desc = trait_obj["description"]

    system = f"""You are a member of the English essay writing test evaluation committee.
Four teachers will be provided with a [Prompt] and an [Essay] written by a student in response to the [Prompt].
Each teacher will score the essay based on different dimensions of writing quality.
Your specific responsibility is to score the essay in terms of "{trait}". {desc}
Focus on the content of the [Essay] and the [Scoring Rubric] to determine the score."""

    user = f"""[Prompt]
{prompt_text}
(end of [Prompt])

[Note]
{ASAP_NOTE}
(end of [Note])

[Essay]
{essay}
(end of [Essay])

Q. List the quotations from the [Essay] that are relevant to "{trait}" and evaluate whether each quotation is well-written or not."""

    return [{"role": "system", "content": system}, {"role": "user", "content": user}]

def build_score_messages(prompt_text, essay, trait_obj, quote_response):
    trait = trait_obj["trait"]
    rubric = trait_obj["rubric"]

    messages = build_quote_messages(prompt_text, essay, trait_obj)
    messages.append({"role": "assistant", "content": quote_response})

    scoring_user = f"""[Scoring Rubric]
**{trait}**:
{rubric}
(end of [Scoring Rubric])

Q. Based on the [Scoring Rubric] and the quotations you found, how would you rate the "{trait}" of this essay?
Assign a score from 0 to 10, strictly following the [Output Format] below.

[Output Format]
Score: <score>insert ONLY the numeric score from 0 to 10 here</score>
(End of [Output Format])"""

    messages.append({"role": "user", "content": scoring_user})
    return messages

# =========================
# 9. Batched generation and parsing
# =========================
@torch.inference_mode()
def chat_generate_batch(messages_list, max_new_tokens=128, temperature=0.1):
    texts = [
        tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        for messages in messages_list
    ]
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_LENGTH,
    ).to(model.device)

    input_len = inputs["input_ids"].shape[1]
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True if temperature > 0 else False,
        temperature=temperature if temperature > 0 else None,
        repetition_penalty=REPETITION_PENALTY,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    generations = []
    for i in range(outputs.shape[0]):
        gen_ids = outputs[i][input_len:]
        generations.append(tokenizer.decode(gen_ids, skip_special_tokens=True).strip())

    return generations

def parse_score(text):
    m = re.search(r"<score>\s*([0-9]+(?:\.[0-9]+)?)\s*</score>", text, flags=re.I)
    if m:
        return float(m.group(1))

    m = re.search(r"score\s*[:\-]\s*([0-9]+(?:\.[0-9]+)?)", text, flags=re.I)
    if m:
        return float(m.group(1))

    nums = re.findall(r"(?<!\d)(?:10(?:\.0+)?|[0-9](?:\.[0-9]+)?)(?!\d)", text)
    if nums:
        return float(nums[-1])

    return np.nan

def clip_score_0_10(x):
    if pd.isna(x):
        return np.nan
    return float(np.clip(x, 0, 10))

def batched(iterable, batch_size):
    batch = []
    for item in iterable:
        batch.append(item)
        if len(batch) == batch_size:
            yield batch
            batch = []
    if batch:
        yield batch

# =========================
# 10. Run Full MTS inference with checkpoint
# =========================
raw_rows = []
checkpoint_path = OUTPUT_DIR / "raw_trait_scores_full_mts_paper_split.csv"

if checkpoint_path.exists():
    raw_done = pd.read_csv(checkpoint_path)
    done_keys = set(zip(raw_done["split"], raw_done["essay_id"], raw_done["trait"]))
    raw_rows = raw_done.to_dict("records")
    print("Resuming from checkpoint:", checkpoint_path, "done:", len(done_keys))
else:
    done_keys = set()

jobs = []
for _, row in score_df.iterrows():
    split = row["split"]
    essay_id = int(row["essay_id"])

    for trait_obj in get_traits(row):
        trait = trait_obj["trait"]
        key = (split, essay_id, trait)
        if key in done_keys:
            continue

        jobs.append({
            "split": split,
            "essay_id": essay_id,
            "essay_set": int(row["essay_set"]),
            "gold": int(row["domain1_score"]),
            "essay": row["essay"],
            "prompt_text": get_prompt_text(row),
            "trait_obj": trait_obj,
            "trait": trait,
        })

print("Remaining jobs:", len(jobs))
print("Each job = one essay-trait. Full MTS uses 2 generations per job.")
print("Approx remaining generations:", len(jobs) * 2)
print("Inference batch size:", INFER_BATCH_SIZE)

num_batches = (len(jobs) + INFER_BATCH_SIZE - 1) // INFER_BATCH_SIZE

for batch_jobs in tqdm(batched(jobs, INFER_BATCH_SIZE), total=num_batches):
    try:
        quote_messages_list = [
            build_quote_messages(job["prompt_text"], job["essay"], job["trait_obj"])
            for job in batch_jobs
        ]

        quote_responses = chat_generate_batch(
            quote_messages_list,
            max_new_tokens=MAX_NEW_TOKENS_QUOTE,
            temperature=TEMPERATURE,
        )

        score_messages_list = [
            build_score_messages(job["prompt_text"], job["essay"], job["trait_obj"], quote_response)
            for job, quote_response in zip(batch_jobs, quote_responses)
        ]

        score_responses = chat_generate_batch(
            score_messages_list,
            max_new_tokens=MAX_NEW_TOKENS_SCORE,
            temperature=TEMPERATURE,
        )

        for job, quote_response, score_response in zip(batch_jobs, quote_responses, score_responses):
            raw_rows.append({
                "split": job["split"],
                "essay_id": job["essay_id"],
                "essay_set": job["essay_set"],
                "gold": job["gold"],
                "trait": job["trait"],
                "trait_score_raw": clip_score_0_10(parse_score(score_response)),
                "quote_response": quote_response,
                "score_response": score_response,
            })

    except torch.cuda.OutOfMemoryError:
        print("CUDA OOM. Reduce INFER_BATCH_SIZE to 1 or 2 and rerun.")
        torch.cuda.empty_cache()
        raise

    except Exception as e:
        err = repr(e)
        print("Batch error:", err)

        if ("CUDA" in err) or ("cuda" in err) or ("illegal memory access" in err):
            print("CUDA context may be corrupted. Restart runtime and reduce INFER_BATCH_SIZE.")
            raise

        for job in batch_jobs:
            raw_rows.append({
                "split": job["split"],
                "essay_id": job["essay_id"],
                "essay_set": job["essay_set"],
                "gold": job["gold"],
                "trait": job["trait"],
                "trait_score_raw": np.nan,
                "quote_response": f"ERROR: {repr(e)}",
                "score_response": f"ERROR: {repr(e)}",
            })

    pd.DataFrame(raw_rows).to_csv(checkpoint_path, index=False)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

raw_df = pd.DataFrame(raw_rows)
raw_df.to_csv(checkpoint_path, index=False)

print("Saved:", checkpoint_path)
display(raw_df.head())
print("NaN scores:", raw_df["trait_score_raw"].isna().sum(), "/", len(raw_df))

# =========================
# 11. Aggregate traits + paper scaling per split and prompt
# =========================
def scale_prompt_scores(group, score_col="trait_avg"):
    p = int(group["essay_set"].iloc[0])
    a, b = ASAP_SCORE_RANGES[p]

    vals = group[score_col].astype(float).to_numpy()

    if np.isnan(vals).any():
        med = np.nanmedian(vals)
        if np.isnan(med):
            med = 5.0
        vals = np.where(np.isnan(vals), med, vals)

    # Outlier clipping using Q1 and Q3.
    q1 = np.percentile(vals, 25)
    q3 = np.percentile(vals, 75)
    iqr = q3 - q1
    vmin = q1 - 1.5 * iqr
    vmax = q3 + 1.5 * iqr
    clipped = np.clip(vals, vmin, vmax)

    # Min-max scaling to prompt-specific target range.
    smin, smax = clipped.min(), clipped.max()
    if abs(smax - smin) < 1e-8:
        scaled = np.full_like(clipped, fill_value=(a + b) / 2, dtype=float)
    else:
        scaled = a + (clipped - smin) * (b - a) / (smax - smin)

    rounded = np.rint(scaled).astype(int)
    rounded = np.clip(rounded, a, b)

    out = group.copy()
    out["trait_avg_filled"] = vals
    out["trait_avg_clipped"] = clipped
    out["pred_scaled"] = scaled
    out["pred_rounded"] = rounded
    return out

agg_df = (
    raw_df
    .groupby(["split", "essay_id", "essay_set", "gold"], as_index=False)["trait_score_raw"]
    .mean()
    .rename(columns={"trait_score_raw": "trait_avg"})
)

pred_df = (
    agg_df
    .groupby(["split", "essay_set"], group_keys=False)
    .apply(scale_prompt_scores)
    .reset_index(drop=True)
)

pred_path = OUTPUT_DIR / "full_mts_predictions_paper_split_scaled.csv"
pred_df.to_csv(pred_path, index=False)

print("Saved:", pred_path)
display(pred_df.head())

# =========================
# 12. QWK without sklearn
# =========================
def qwk(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)

    if len(y_true) == 0:
        return np.nan

    min_rating = min(y_true.min(), y_pred.min())
    max_rating = max(y_true.max(), y_pred.max())
    num_ratings = max_rating - min_rating + 1

    if num_ratings <= 1:
        return 0.0

    yt = y_true - min_rating
    yp = y_pred - min_rating

    O = np.zeros((num_ratings, num_ratings), dtype=float)
    for a, b in zip(yt, yp):
        O[a, b] += 1

    true_hist = np.bincount(yt, minlength=num_ratings).astype(float)
    pred_hist = np.bincount(yp, minlength=num_ratings).astype(float)
    E = np.outer(true_hist, pred_hist) / len(y_true)

    W = np.zeros((num_ratings, num_ratings), dtype=float)
    denom = (num_ratings - 1) ** 2
    for i in range(num_ratings):
        for j in range(num_ratings):
            W[i, j] = ((i - j) ** 2) / denom

    numerator = np.sum(W * O)
    denominator = np.sum(W * E)

    if denominator == 0:
        return 0.0

    return 1.0 - numerator / denominator

# =========================
# 13. Evaluate by prompt, then average over prompts
# =========================
rows = []
for (split, p), g in pred_df.groupby(["split", "essay_set"]):
    rows.append({
        "split": split,
        "essay_set": int(p),
        "n": len(g),
        "gold_min": int(g["gold"].min()),
        "gold_max": int(g["gold"].max()),
        "pred_min": int(g["pred_rounded"].min()),
        "pred_max": int(g["pred_rounded"].max()),
        "qwk": qwk(g["gold"], g["pred_rounded"]),
    })

eval_by_prompt = pd.DataFrame(rows).sort_values(["split", "essay_set"])

eval_avg = (
    eval_by_prompt
    .groupby("split")
    .agg(
        avg_qwk_over_prompts=("qwk", "mean"),
        num_prompts=("essay_set", "nunique"),
        num_essays=("n", "sum"),
    )
    .reset_index()
)

# Table-like format: P1-P8 + Avg.
table_rows = []
for split, g in eval_by_prompt.groupby("split"):
    row = {"split": split}
    for p in range(1, 9):
        vals = g.loc[g["essay_set"] == p, "qwk"].values
        row[f"P{p}"] = vals[0] if len(vals) else np.nan
    row["Avg"] = np.nanmean([row[f"P{p}"] for p in range(1, 9)])
    table_rows.append(row)

eval_table = pd.DataFrame(table_rows)

eval_by_prompt_path = OUTPUT_DIR / "full_mts_mistral7b_qwk_by_prompt_paper_split.csv"
eval_avg_path = OUTPUT_DIR / "full_mts_mistral7b_qwk_average_paper_split.csv"
eval_table_path = OUTPUT_DIR / "full_mts_mistral7b_qwk_table_paper_split.csv"

eval_by_prompt.to_csv(eval_by_prompt_path, index=False)
eval_avg.to_csv(eval_avg_path, index=False)
eval_table.to_csv(eval_table_path, index=False)

print("QWK by prompt")
display(eval_by_prompt)

print("Paper-style table")
display(eval_table)

print("Average QWK")
display(eval_avg)

print("Saved:")
print(eval_by_prompt_path)
print(eval_avg_path)
print(eval_table_path)


Loaded: /content/asap_train.csv (9084, 4)
Loaded: /content/asap_val.csv (1296, 4)
Loaded: /content/asap_test.csv (2596, 4)
Reconstructed full ASAP from split files: (12976, 4)
Full ASAP: (12976, 4)
Paper-style test: (1297, 5)
Unused trainval: (11679, 5)


,split,essay_set,count,min,max,mean
0,test,1,178,4,12,8.623596
1,test,2,180,1,6,3.388889
2,test,3,173,0,3,1.803468
3,test,4,177,0,3,1.514124
4,test,5,180,0,4,2.555556
5,test,6,180,0,4,2.600000
6,test,7,157,4,24,16.382166
7,test,8,72,20,55,36.333333
8,trainval_unused,1,1605,2,12,8.517757
9,trainval_unused,2,1620,1,6,3.418519


Rows to score: 1297


,split,essay_set,n
0,test,1,178
1,test,2,180
2,test,3,173
3,test,4,177
4,test,5,180
5,test,6,180
6,test,7,157
7,test,8,72


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


...

Paper-style table


,split,P1,P2,P3,P4,P5,P6,P7,P8,Avg
0,test,0.56427,0.550926,0.549607,0.548015,0.547104,0.558513,0.577545,0.347304,0.53041


Average QWK


,split,avg_qwk_over_prompts,num_prompts,num_essays
0,test,0.53041,8,1297


Saved:
/content/mts_mistral7b_full_paper_split_outputs/full_mts_mistral7b_qwk_by_prompt_paper_split.csv
/content/mts_mistral7b_full_paper_split_outputs/full_mts_mistral7b_qwk_average_paper_split.csv
/content/mts_mistral7b_full_paper_split_outputs/full_mts_mistral7b_qwk_table_paper_split.csv


## Recommended run order

Để tránh tốn GPU quá nhiều, nên chạy theo thứ tự:

1. Smoke test:
```python
PROMPTS_TO_SCORE = [1]
MAX_ESSAYS_PER_PROMPT_PER_SPLIT = 5
INFER_BATCH_SIZE = 1
```

2. Nếu ổn, chạy subset:
```python
PROMPTS_TO_SCORE = None
MAX_ESSAYS_PER_PROMPT_PER_SPLIT = 20
INFER_BATCH_SIZE = 2
```

3. Full paper-style test:
```python
PROMPTS_TO_SCORE = None
MAX_ESSAYS_PER_PROMPT_PER_SPLIT = None
INFER_BATCH_SIZE = 2 hoặc 4
```

Full MTS tốn khoảng:
- số essay test ASAP paper-style: khoảng 1,300
- 4 traits / essay
- 2 generations / trait
- tổng khoảng 10,000+ generations

Nếu bị OOM, giảm `INFER_BATCH_SIZE` xuống 1 hoặc 2.
